# Auto-Labeling — Pretrained YOLOv8 → YOLO .txt files

## 1 · Imports & schema decision

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from collections import Counter
import random
import json

import cv2
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.auto_label import auto_label_directory, DEFAULT_CONF
from src.dataset_builder import decide_class_schema, build_dataset

FRAMES_DIR  = ROOT / 'data' / 'frames'
LABELS_DIR  = ROOT / 'data' / 'labels'
DATASET_DIR = ROOT / 'data' / 'dataset'
CONFIG_PATH = ROOT / 'configs' / 'dataset.yaml'
REPORT_PATH = ROOT / 'outputs' / 'labeling_report.json'


### Decide the class schema

In [ ]:
schema, decision = decide_class_schema(FRAMES_DIR)
print('Decision:', decision)
print('Chosen schema:', schema)


## 2 · Auto-label all frames

In [ ]:
report = auto_label_directory(
    frames_root=FRAMES_DIR,
    labels_root=LABELS_DIR,
    schema=schema,
    conf=DEFAULT_CONF,
    device='cuda:0',           # falls back to CPU automatically if unavailable
    report_path=REPORT_PATH,
)
print(json.dumps(report.as_dict(), indent=2))


## 3 · Class balance

In [ ]:
counts = report.boxes_per_class
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.keys(), counts.values())
ax.set_ylabel('boxes'); ax.set_title('Auto-labeled boxes per class')
for k, v in counts.items():
    ax.text(k, v, str(v), ha='center', va='bottom')
plt.tight_layout(); plt.show()


## 4 · Confidence distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for cname, confs in report.confidences_per_class.items():
    if confs:
        ax.hist(confs, bins=20, alpha=0.5, label=f'{cname} (n={len(confs)})')
ax.set_xlabel('confidence'); ax.set_ylabel('count')
ax.set_title('Auto-label confidence distribution per class')
ax.legend(); plt.tight_layout(); plt.show()


## 5 · Visual spot-check — 20 random labeled frames

In [ ]:
COLORS = {
    'fire':       (255, 60,  60),
    'smoke':      (255, 180, 60),
    'person':     (60,  220, 60),
    'fire_smoke': (255, 60,  60),
}
INV_SCHEMA = {v: k for k, v in schema.items()}

labeled = sorted(LABELS_DIR.rglob('*.txt'))
random.seed(0)
sample = random.sample(labeled, min(20, len(labeled)))

rows = 5; cols = 4
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
for ax, lbl in zip(axes.ravel(), sample):
    rel = lbl.relative_to(LABELS_DIR).with_suffix('.jpg')
    img_path = FRAMES_DIR / rel
    if not img_path.exists():
        ax.axis('off'); continue
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    for line in lbl.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5: continue
        cid = int(parts[0]); cx, cy, bw, bh = map(float, parts[1:5])
        x1 = int((cx - bw / 2) * w); y1 = int((cy - bh / 2) * h)
        x2 = int((cx + bw / 2) * w); y2 = int((cy + bh / 2) * h)
        cname = INV_SCHEMA.get(cid, str(cid))
        color = COLORS.get(cname, (200, 200, 200))
        cv2.rectangle(img, (x1, y1), (x2, y2), color[::-1], 2)
        cv2.putText(img, cname, (x1, max(0, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color[::-1], 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(rel.as_posix(), fontsize=7); ax.axis('off')
for ax in axes.ravel()[len(sample):]: ax.axis('off')
plt.tight_layout(); plt.show()


## 6 · Build the train/val/test dataset

In [ ]:
summary = build_dataset(
    frames_root=FRAMES_DIR,
    labels_root=LABELS_DIR,
    dataset_root=DATASET_DIR,
    config_path=CONFIG_PATH,
    schema=schema,
)
summary
